## Balancing PAN25 Training and Testing Data

In [6]:
import pandas as pd
import json

# ----------------------------------------
# Function to balance dataset
# ----------------------------------------
def balance_dataset(input_file, output_file):

    # -------- Load JSONL --------
    data = []

    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))

    df = pd.DataFrame(data)

    balanced_data = []

    # -------- Find minimum HUMAN count across genres --------
    human_counts = df[df["label"] == 0]["genre"].value_counts()
    target_per_label = human_counts.min()

    print(f"\nProcessing: {input_file}")
    print(f"Target per label per genre: {target_per_label}")

    # -------- Process per Genre --------
    for genre in df["genre"].unique():

        genre_df = df[df["genre"] == genre]

        human_df = genre_df[genre_df["label"] == 0]
        ai_df = genre_df[genre_df["label"] == 1]

        # -------- Sample Human --------
        sampled_human = human_df.sample(
            n=target_per_label,
            random_state=42
        )

        # -------- Proportional Sampling for AI --------
        model_counts = ai_df["model"].value_counts()
        total_ai = len(ai_df)

        sampled_ai_parts = []

        for model, count in model_counts.items():

            model_subset = ai_df[ai_df["model"] == model]

            proportion = count / total_ai

            sample_size = int(round(
                proportion * target_per_label
            ))

            sample_size = min(sample_size, len(model_subset))

            if sample_size > 0:
                sampled = model_subset.sample(
                    n=sample_size,
                    random_state=42
                )

                sampled_ai_parts.append(sampled)

        sampled_ai_df = pd.concat(sampled_ai_parts)

        # -------- Adjustment --------
        if len(sampled_ai_df) > target_per_label:

            sampled_ai_df = sampled_ai_df.sample(
                n=target_per_label,
                random_state=42
            )

        elif len(sampled_ai_df) < target_per_label:

            remaining = ai_df.drop(sampled_ai_df.index)

            extra_needed = target_per_label - len(sampled_ai_df)

            extra = remaining.sample(
                n=extra_needed,
                random_state=42
            )

            sampled_ai_df = pd.concat(
                [sampled_ai_df, extra]
            )

        # -------- Combine --------
        genre_final = pd.concat(
            [sampled_human, sampled_ai_df]
        )

        balanced_data.append(genre_final)

    # -------- Final Dataset --------
    final_df = pd.concat(balanced_data)

    final_df = final_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    # -------- Save --------
    with open(output_file, "w", encoding="utf-8") as f:

        for _, row in final_df.iterrows():
            f.write(json.dumps(row.to_dict()) + "\n")

    print(f"Saved: {output_file}")


# ----------------------------------------
# FILE LOOP
# ----------------------------------------
files = {
    "pan25-generative-ai-detection-task1-train\\train.jsonl": "balanced_train_equal_genres.jsonl",
    "pan25-generative-ai-detection-task1-train\\val.jsonl": "balanced_data\\testing.jsonl"
}

for input_file, output_file in files.items():
    balance_dataset(input_file, output_file)


Processing: pan25-generative-ai-detection-task1-train\train.jsonl
Target per label per genre: 870
Saved: balanced_train_equal_genres.jsonl

Processing: pan25-generative-ai-detection-task1-train\val.jsonl
Target per label per genre: 132
Saved: balanced_data\testing.jsonl


## Train-Val Split

In [5]:
# -----------------------------------
# Load balanced 870 dataset
# -----------------------------------
data = []
with open("balanced_train_equal_genres.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

train_parts = []
val_parts = []

VAL_SIZE = 132
RANDOM_STATE = 42

# -----------------------------------
# Split per Genre + Label
# -----------------------------------
for genre in df["genre"].unique():

    genre_df = df[df["genre"] == genre]

    for label in [0, 1]:   # 0 = Human, 1 = AI

        subset = genre_df[genre_df["label"] == label]

        # -----------------------------------
        # HUMAN CLASS
        # -----------------------------------
        if label == 0:
            val_subset = subset.sample(
                n=VAL_SIZE,
                random_state=RANDOM_STATE
            )

            train_subset = subset.drop(val_subset.index)

        # -----------------------------------
        # AI CLASS (preserve model proportions)
        # -----------------------------------
        else:
            model_counts = subset["model"].value_counts()
            total_ai = len(subset)

            val_ai_parts = []

            for model, count in model_counts.items():

                model_subset = subset[subset["model"] == model]

                proportion = count / total_ai
                sample_n = int(round(proportion * VAL_SIZE))
                sample_n = min(sample_n, len(model_subset))

                if sample_n > 0:
                    sampled = model_subset.sample(
                        n=sample_n,
                        random_state=RANDOM_STATE
                    )
                    val_ai_parts.append(sampled)

            val_subset = pd.concat(val_ai_parts)

            # -----------------------------------
            # Adjust exactly 132 samples
            # -----------------------------------
            if len(val_subset) > VAL_SIZE:
                val_subset = val_subset.sample(
                    n=VAL_SIZE,
                    random_state=RANDOM_STATE
                )

            elif len(val_subset) < VAL_SIZE:
                remaining = subset.drop(val_subset.index)
                extra_needed = VAL_SIZE - len(val_subset)

                extra = remaining.sample(
                    n=extra_needed,
                    random_state=RANDOM_STATE
                )

                val_subset = pd.concat([val_subset, extra])

            train_subset = subset.drop(val_subset.index)

        # -----------------------------------
        # Store splits
        # -----------------------------------
        train_parts.append(train_subset)
        val_parts.append(val_subset)

# -----------------------------------
# Final datasets
# -----------------------------------
train_df = pd.concat(train_parts).sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

val_df = pd.concat(val_parts).sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# -----------------------------------
# Save train.jsonl
# -----------------------------------
with open("balanced_data\\training.jsonl", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")

# -----------------------------------
# Save val.jsonl
# -----------------------------------
with open("balanced_data\\validation.jsonl", "w", encoding="utf-8") as f:
    for _, row in val_df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")

# -----------------------------------
# Summary
# -----------------------------------
print("✅ Saved:")
print("training.jsonl")
print("validation.jsonl")

print("\nTrain shape:", train_df.shape)
print("Val shape:", val_df.shape)

print("\nValidation counts by Genre + Label:")
print(val_df.groupby(["genre", "label"]).size())

print("\nTraining counts by Genre + Label:")
print(train_df.groupby(["genre", "label"]).size())

✅ Saved:
training.jsonl
validation.jsonl

Train shape: (4428, 5)
Val shape: (792, 5)

Validation counts by Genre + Label:
genre    label
essays   0        132
         1        132
fiction  0        132
         1        132
news     0        132
         1        132
dtype: int64

Training counts by Genre + Label:
genre    label
essays   0        738
         1        738
fiction  0        738
         1        738
news     0        738
         1        738
dtype: int64
